<a href="https://colab.research.google.com/github/HudaSaffo/fashion-recommendation-system/blob/main/dataset_ingestion_and_caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets pandas pyarrow

Dataset loading from Hugging Face

In [ ]:
from datasets import load_dataset
import pandas as pd
import gc

print("1. Loading outfit mapping data (using 'train' split)...")
outfit_dataset = load_dataset("owj0421/polyvore-outfits", "nondisjoint_default", split="train", streaming=True)
# Take 1,500 outfits
mapping_sample = list(outfit_dataset.take(1500))
df_mapping = pd.DataFrame(mapping_sample)

print("2. Flattening outfit structures...")
flattened_items = []
for index, row in df_mapping.iterrows():
    for item in row['items']:
        flattened_items.append({
            'set_id': str(row['set_id']),
            'item_id': str(item['item_id']),
            'item_index_in_set': int(item['index'])
        })
outfit_df = pd.DataFrame(flattened_items)

# Create a fast-lookup set of unique IDs we actually care about
valid_item_ids = set(outfit_df['item_id'].unique())

# Clean up memory immediately
del mapping_sample, df_mapping
gc.collect()

print("3. Streaming product metadata (RAM-optimized stream)...")
meta_dataset = load_dataset("owj0421/polyvore", split="data", streaming=True)

meta_list = []
for i, record in enumerate(meta_dataset):
    # 1. RAM TRICK: Delete the massive PIL image object from the dictionary immediately
    if 'image' in record:
        del record['image']

    # 2. Only append to memory if the item is in our active outfit set
    record_id = str(record['item_id'])
    if record_id in valid_item_ids:
        meta_list.append(record)

    if i >= 120000:
        break

meta_df = pd.DataFrame(meta_list)
meta_df['item_id'] = meta_df['item_id'].astype(str)

final_catalog = pd.merge(outfit_df, meta_df, on='item_id', how='inner')
final_catalog = final_catalog.drop_duplicates(subset=['item_id'])

final_catalog.to_parquet("catalog_sample.parquet", index=False)
final_catalog.to_csv("catalog_sample.csv", index=False)

print(f"\n Combined catalog created with {len(final_catalog)} unique items.")
print("Saved data fields:", final_catalog.columns.tolist())

In [ ]:
final_catalog.head(3)

Google Drive mounting

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
import gc

CACHE_DIR = "/content/drive/MyDrive/polyvore_image_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

try:
    df = pd.read_parquet("catalog_sample.parquet")
except Exception as e:
    df = pd.read_csv("catalog_sample.csv")

valid_item_ids = set(df['item_id'].astype(str).unique())

# Check how many images we already saved permanently
existing_images = len([f for f in os.listdir(CACHE_DIR) if f.endswith('.jpg')])
print(f"Permanent Drive Folder: {existing_images} / {len(valid_item_ids)} images already saved.")

if existing_images >= len(valid_item_ids):
    print("All images are already safe in the permanent Google Drive")
else:
    print("\nStreaming master dataset to save remaining images permanently to Google Drive")
    meta_dataset = load_dataset("owj0421/polyvore", split="data", streaming=True)

    success_count = existing_images

    for i, record in enumerate(meta_dataset):
        record_id = str(record['item_id'])

        if record_id in valid_item_ids:
            image_path = os.path.join(CACHE_DIR, f"{record_id}.jpg")

            # Skip if already saved in Drive
            if os.path.exists(image_path):
                continue

            if 'image' in record and record['image'] is not None:
                try:
                    record['image'].save(image_path, "JPEG")
                    success_count += 1

                    if success_count % 500 == 0:
                        print(f"Saved {success_count} images permanently")
                except:
                    pass

        if 'image' in record:
            del record['image']

        if i >= 120000 or success_count >= len(valid_item_ids):
            break

    print(f"Total secure images sitting in Google Drive: {success_count}")